# 🎬 Short Builder — 自動短影音製作(完全免費・線上執行)

餵一個 **YouTube / 其他影片網址**,這個筆記本會自動:

1. 下載影片(yt-dlp)
2. 語音辨識 + 逐字時間軸(faster-whisper)
3. 自動挑出最精華的片段
4. 剪成 **9:16 直式短影音**
5. 燒錄字幕
6.(可選)用 edge-tts 重新配音

**完全免費**:在 Google Colab 上執行,連帳號付費都不用。建議先到
`執行階段 → 變更執行階段類型 → T4 GPU`,語音辨識會快很多(CPU 也能跑,只是較慢)。

> 使用方式:由上往下,每個格子按 ▶️ 執行即可。


## 1️⃣ 安裝套件(約 1–2 分鐘)

In [ ]:
#@title 安裝 ffmpeg 與 Python 套件 { display-mode: "form" }
import subprocess, sys
print("安裝 ffmpeg…")
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "fonts-noto-cjk"], check=True)
print("安裝 Python 套件…")
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "yt-dlp", "faster-whisper", "edge-tts"], check=True)

import torch
GPU = torch.cuda.is_available()
print("\\n✅ 安裝完成 ·", "偵測到 GPU 🚀" if GPU else "使用 CPU(較慢,建議改用 T4 GPU 執行階段)")

## 2️⃣ 載入製作引擎(直接執行即可,不用改)

In [ ]:
#@title 載入 Short Builder 引擎 { display-mode: "form" }
import asyncio, re, shutil, subprocess
from pathlib import Path

def hms(t):
    h, rem = divmod(int(t), 3600); m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d},{int((t-int(t))*1000):03d}"

def acquire_video(source, workdir):
    local = Path(source)
    if local.exists():
        print(f"[1/5] 使用上傳檔案:{local}")
        return local.resolve(), local.stem
    import yt_dlp
    print(f"[1/5] 下載影片:{source}")
    opts = {"format": "bv*[height<=1080][ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b",
            "outtmpl": str(workdir / "source.%(ext)s"), "merge_output_format": "mp4",
            "noplaylist": True, "quiet": True, "no_warnings": True}
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(source, download=True)
    path = next(workdir.glob("source.*"))
    print("      完成:", info.get("title", "short"))
    return path, info.get("title", "short")

def transcribe(path, model_size, lang):
    from faster_whisper import WhisperModel
    import torch
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ct = "float16" if dev == "cuda" else "int8"
    print(f"[2/5] 語音辨識(whisper-{model_size} / {dev},首次會下載模型)…")
    model = WhisperModel(model_size, device=dev, compute_type=ct)
    raw, info = model.transcribe(str(path), language=None if lang == "auto" else lang,
                                 word_timestamps=True, vad_filter=True)
    segs = []
    for s in raw:
        text = s.text.strip()
        if text:
            segs.append({"start": s.start, "end": s.end, "text": text,
                         "words": [{"w": w.word, "s": w.start, "e": w.end} for w in (s.words or [])]})
    print(f"      語言:{info.language},共 {len(segs)} 句")
    return segs

def _score(window):
    dur = window[-1]["end"] - window[0]["start"]
    if dur <= 0: return 0.0
    chars = sum(len(s["text"]) for s in window)
    bonus = 0.0
    for s in window:
        if re.search(r"[!?!?]", s["text"]): bonus += 2.0
        if re.search(r"\d", s["text"]): bonus += 0.5
    return chars / dur + bonus * 0.3

def pick_highlights(segs, duration, n_clips):
    if not segs: return []
    cand = []
    for i in range(len(segs)):
        j = i
        while j + 1 < len(segs) and segs[j+1]["end"] - segs[i]["start"] <= duration * 1.1:
            j += 1
        w = segs[i:j+1]
        if w[-1]["end"] - w[0]["start"] < min(duration*0.5, 10): continue
        cand.append((_score(w), w[0]["start"], w[-1]["end"], w))
    if not cand: return [(segs[0]["start"], segs[-1]["end"], segs)]
    cand.sort(key=lambda c: -c[0])
    picked = []
    for sc, st, en, w in cand:
        if any(not (en <= ps or st >= pe) for _, ps, pe, _ in picked): continue
        picked.append((sc, st, en, w))
        if len(picked) >= n_clips: break
    picked.sort(key=lambda c: c[1])
    return [(s, e, w) for _, s, e, w in picked]

def build_srt(window, clip_start, srt_path, max_chars=14, max_dur=2.2):
    words = [w for s in window for w in s["words"]] or \
            [{"w": s["text"], "s": s["start"], "e": s["end"]} for s in window]
    chunks, cur = [], []
    for w in words:
        cur.append(w)
        t = "".join(x["w"] for x in cur).strip()
        if len(t) >= max_chars or cur[-1]["e"] - cur[0]["s"] >= max_dur:
            chunks.append(cur); cur = []
    if cur: chunks.append(cur)
    lines = []
    for i, c in enumerate(chunks, 1):
        st = max(0.0, c[0]["s"] - clip_start)
        en = max(st + 0.3, c[-1]["e"] - clip_start)
        lines.append(f"{i}\n{hms(st)} --> {hms(en)}\n{''.join(x['w'] for x in c).strip()}\n")
    srt_path.write_text("\n".join(lines), encoding="utf-8")
    return len(chunks)

SUB_STYLE = ("FontName=Noto Sans CJK TC,FontSize=15,Bold=1,PrimaryColour=&H00FFFFFF,"
             "OutlineColour=&H00000000,BorderStyle=1,Outline=2,Shadow=1,MarginV=60,Alignment=2")

async def _tts(text, voice, path):
    import edge_tts
    await edge_tts.Communicate(text, voice).save(str(path))

def add_voiceover(clip, text, voice, keep_audio, workdir, out_path):
    vo = workdir / "vo.mp3"
    print(f"  [tts] 配音({voice})…")
    asyncio.run(_tts(text, voice, vo))
    if keep_audio:
        f = ("[0:a]volume=0.15[bg];[1:a]apad[vo];[bg][vo]amix=inputs=2:duration=first[a]")
        args = ["-i", str(clip), "-i", str(vo), "-filter_complex", f,
                "-map", "0:v", "-map", "[a]", "-c:v", "copy", "-c:a", "aac", str(out_path)]
    else:
        args = ["-i", str(clip), "-i", str(vo), "-map", "0:v", "-map", "1:a",
                "-af", "apad", "-c:v", "copy", "-c:a", "aac", "-shortest", str(out_path)]
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y"] + args, check=True)

def build(source, duration=45, clips=1, start=None, end=None, lang="auto",
          model="small", voice="", keep_audio=False, subs=True, crop=True, out="output"):
    out_dir = Path(out).resolve(); out_dir.mkdir(parents=True, exist_ok=True)
    workdir = out_dir / ".work"; workdir.mkdir(exist_ok=True)
    video, title = acquire_video(source, workdir)
    segs = transcribe(video, model, lang) if (subs or voice or start is None) else []
    if start is not None:
        e = end if end is not None else start + duration
        window = [s for s in segs if s["end"] > start and s["start"] < e]
        segments = [(start, e, window)]
        print(f"[3/5] 手動片段 {start:.1f}s – {e:.1f}s")
    else:
        segments = pick_highlights(segs, duration, clips)
        print(f"[3/5] 自動挑出 {len(segments)} 段精華:")
        for s, e, w in segments:
            print(f"      {s:7.1f}s – {e:7.1f}s  「{(w[0]['text'][:20] if w else '')}…」")
    safe = re.sub(r"[^\w一-鿿-]+", "_", title)[:40].strip("_") or "short"
    results = []
    for idx, (s, e, window) in enumerate(segments, 1):
        print(f"[4/5] 剪輯第 {idx}/{len(segments)} 支…")
        pad = max(0.0, s - 0.2); raw = workdir / f"clip_{idx}.mp4"
        vf = []
        if crop: vf.append("scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920")
        if subs and window:
            srt = workdir / f"clip_{idx}.srt"; build_srt(window, pad, srt)
            vf.append(f"subtitles={srt.name}:force_style='{SUB_STYLE}'")
        cmd = ["-ss", f"{pad:.2f}", "-to", f"{e+0.3:.2f}", "-i", str(video)]
        if vf: cmd += ["-vf", ",".join(vf)]
        cmd += ["-c:v", "libx264", "-preset", "medium", "-crf", "20", "-c:a", "aac",
                "-b:a", "160k", "-movflags", "+faststart", str(raw)]
        subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y"] + cmd,
                       check=True, cwd=workdir)
        final = out_dir / f"{safe}_short{idx}.mp4"
        if voice and window:
            print(f"[5/5] 配音第 {idx} 支…")
            add_voiceover(raw, "".join(x["text"] for x in window), voice, keep_audio, workdir, final)
        else:
            shutil.move(str(raw), final)
        print(f"  ✓ 輸出:{final}"); results.append(final)
    print(f"\n🎉 完成!共 {len(results)} 支短影音。")
    return results

print("✅ 引擎已載入")

## 3️⃣ 製作短影音

填好下面的欄位後執行。第一次跑 whisper 會下載模型(small 約 460MB)。

In [ ]:
#@title ▶️ 設定並製作 { display-mode: "form" }
#@markdown ### 來源
影片網址 = "https://www.youtube.com/watch?v=" #@param {type:"string"}
#@markdown 若要改用上傳的檔案,把檔名填到下面(留空就用上面的網址)
上傳檔名 = "" #@param {type:"string"}

#@markdown ### 短片設定
每支秒數 = 45 #@param {type:"slider", min:15, max:90, step:5}
產生幾支 = 1 #@param {type:"slider", min:1, max:5, step:1}
裁成直式9比16 = True #@param {type:"boolean"}
加字幕 = True #@param {type:"boolean"}

#@markdown ### 語音辨識
語言 = "auto" #@param ["auto", "zh", "en", "ja", "ko"]
模型大小 = "small" #@param ["tiny", "base", "small", "medium", "large-v3"]

#@markdown ### 配音(留 none 就保留原音)
配音聲音 = "none" #@param ["none", "zh-TW-HsiaoChenNeural", "zh-TW-YunJheNeural", "zh-CN-XiaoxiaoNeural", "en-US-AriaNeural", "ja-JP-NanamiNeural"]
配音時保留原音當背景 = False #@param {type:"boolean"}

#@markdown ### 手動指定片段(可選,填了就不自動挑選)
手動起點秒 = -1 #@param {type:"number"}
手動終點秒 = -1 #@param {type:"number"}

src = 上傳檔名.strip() or 影片網址.strip()
results = build(
    src, duration=每支秒數, clips=產生幾支,
    start=None if 手動起點秒 < 0 else 手動起點秒,
    end=None if 手動終點秒 < 0 else 手動終點秒,
    lang=語言, model=模型大小,
    voice="" if 配音聲音 == "none" else 配音聲音,
    keep_audio=配音時保留原音當背景, subs=加字幕, crop=裁成直式9比16,
)

## 4️⃣ 預覽 & 下載

In [ ]:
#@title 預覽第一支並下載全部 { display-mode: "form" }
from IPython.display import HTML
from base64 import b64encode
from google.colab import files

if results:
    data = b64encode(open(results[0], "rb").read()).decode()
    display(HTML(f'<video width=320 controls src="data:video/mp4;base64,{data}"></video>'))
    for f in results:
        files.download(str(f))
else:
    print("還沒有產生任何短片,請先執行上一個格子。")

---
### 其他免費線上選項
- **Kaggle Notebooks**:同樣免費、也有 GPU,操作方式幾乎相同。
- **HuggingFace Spaces(Gradio)**:可以做成一個貼網址就出片的網頁 App(免費版為 CPU,長片較慢)。需要的話我可以再幫你做這個版本。

> 提醒:請只下載與剪輯你擁有權利或已獲授權的內容。
